In [14]:
#  Use Miniconda
!pip install neo4j
!pip install plotly.express

In [15]:
scenario = "crmscripted"

In [16]:
import plotly.express as px
import pandas as pd
import json
from datetime import datetime

##############################  READ  FILES  ##############################
with open("merged_orders_"+scenario+".json") as f: #Presentations & Orders
    structuredOrders = json.load(f)

# with open("structuredTimeline2.json") as f: # Actions
#     structuredActions = json.load(f)

with open("merged_skills_"+scenario+".json") as f: #Skills
    structuredSkills = json.load(f)


##############################  CHECK PARTICIPANTS  ##############################
participantsList = {}
participantsIds = {}
participantsRoles = {}

# for presentationBlock in structuredOrders['presentationsTimeline']:
#     participant = presentationBlock['speaker']['name']
#     participantsList[participant] = participant


# Get names of participants from orders
for orderBlock in structuredOrders['ordersTimeline']:

    participant = orderBlock['leader']['name']
    if participant not in participantsList and participant != "":
        participantsList[participant] = participant
        participantsIds[participant] = orderBlock['leader']["speaker_id"]
        participantsRoles[participant] = orderBlock['leader']["role"]
    elif participant not in participantsList and participant == "":
        participantsList[participant] = "Unknown"

    follower = orderBlock['follower']
    if follower != "None" and follower != "Unknown" and follower != None:
        follower = orderBlock['follower']['name']
    if follower not in participantsList and follower != "" and follower != "None" and follower != "Unknown" and follower != None:
        participantsList[follower] = follower
        participantsIds[follower] = orderBlock['follower']["speaker_id"]
        participantsRoles[follower] = orderBlock['follower']["role"]
    elif follower not in participantsList and (follower == "" or follower == "None" or follower == "Unknown" or follower == None):
        if follower == None:
          follower = "None"
        participantsList[follower] = "Unknown"


# for actionBlock in structuredActions['actionsTimeline']:
#     participant = actionBlock['speaker']['name']
#     if participant not in participantsList and participant != "":
#         participantsList[participant] = participant
#     elif participant not in participantsList and participant == "":
#         participantsList[participant] = "Unknown"


for skillBlock in structuredSkills['skillsTimeline']:
    participant = skillBlock['speaker']['name']
    if participant not in participantsList and participant != "":
        participantsList[participant] = participant
        participantsIds[participant] = skillBlock['speaker']["speaker_id"]
        participantsRoles[participant] = skillBlock['speaker']["role"]
    elif participant not in participantsList and participant == "":
        participantsList[participant] = "Unknown"
participantsList["None"] = "Unknown"


print("Participants: ")
print(participantsList.values())
print(participantsList.keys())

##############################  REVIEW OF PARTICIPANTS  ##############################
for participant in participantsList.keys():
    if participant != "Unknown" and participant != "" and participant != "None" and participant != "Unknown" and participant != None:
        print("Is "+participant+" a participant? Y/N")
        response = input()
        if response != "Y" and response != "y" and response != "yes" and response != "Yes" and response != "YES":
            print("Introduce the name of the participant to fix the translations: ")
            new_participant = input()
            participantsList[participant] = new_participant
            participantsIds[new_participant] = participantsIds[participant]
            participantsRoles[new_participant] = participantsRoles[participant]


print(participantsList)


# # print("Show presentations? Y/N")
# # show_presentations = input()
# # if show_presentations != "Y" and show_presentations != "y" and show_presentations != "yes" and show_presentations != "Yes" and show_presentations != "YES":
# #   show_presentations = False
# # else:
# #   show_presentations = True

# print("Show orders? Y/N")
# show_orders = input()
# if show_orders != "Y" and show_orders != "y" and show_orders != "yes" and show_orders != "Yes" and show_orders != "YES":
#   show_orders = False
# else:
#   show_orders = True
show_orders = True

# # print("Show actions? Y/N")
# # show_actions = input()
# # if show_actions != "Y" and show_actions != "y" and show_actions != "yes" and show_actions != "Yes" and show_actions != "YES":
# #   show_actions = False
# # else:
# #   show_actions = True

# print("Show non-technical skills? Y/N")
# show_skills = input()
# if show_skills != "Y" and show_skills != "y" and show_skills != "yes" and show_skills != "Yes" and show_skills != "YES":
#   show_skills = False
# else:
#   show_skills = True
show_skills = True


timelineList = []
ordersList = []
skillList = []
# ############ Presentations & Orders ############
# # if show_presentations == True:
# #   print("Showing presentations")
# #   for presentationBlock in structuredOrders['presentationsTimeline']:
# #       start_time = datetime.utcfromtimestamp(presentationBlock['start_timestamp']).strftime('%Y-%m-%d %H:%M:%S.%f')
# #       stop_time = datetime.utcfromtimestamp(presentationBlock['stop_timestamp']).strftime('%Y-%m-%d %H:%M:%S.%f')
# #       speaker_name = presentationBlock['speaker']['name']
# #       actionText = participantsList[speaker_name] + ": presents role " + presentationBlock['speaker']['role']

# #       # Replace fixed participants named
# #       for key,value in participantsList.items():
# #         if key != "":
# #           actionText = actionText.replace(key,value)
# #       timelineList.append(dict(Task=actionText, Start=start_time, Finish=stop_time, Category = "Presentation"))

if show_orders == True:
  print("Showing orders")
  for orderBlock in structuredOrders['ordersTimeline']:
      stop_time = orderBlock['stop_timestamp']
      start_time = orderBlock['start_timestamp']

      if stop_time == "None" or stop_time == None:
          stop_time = float(start_time) + 3.0

      if float(stop_time) < float(start_time) + 1.0:
          stop_time = float(start_time) + 3.0

      stop_time = datetime.utcfromtimestamp(stop_time).strftime('%Y-%m-%d %H:%M:%S.%f')
      start_time = datetime.utcfromtimestamp(orderBlock['start_timestamp']).strftime('%Y-%m-%d %H:%M:%S.%f')

      leader = orderBlock['leader']['name']
      follower = orderBlock['follower']
      # if leader == "":
      #     leader = orderBlock['leader']['speaker_id']
      if follower != "None" and follower != "Unknown" and follower != None:
          follower = orderBlock['follower']['name']
      elif follower == None:
          follower = "None"
      actionText = participantsList[leader] + " to " + participantsList[follower] + ": " + orderBlock['action']

      # Replace fixed participants named
      for key,value in participantsList.items():
        if key != "":
          actionText = actionText.replace(key,value)
      timelineList.append(dict(Task=actionText, Start=start_time, Finish=stop_time, Category = "Order"))
      ordersList.append(dict(Task=actionText, Start=start_time, Finish=stop_time, Category = "Order"))



# ############ Actions ############
# if show_actions == True:
#   print("Showing actions")
#   for actionBlock in structuredActions['actionsTimeline']:
#       stop_time = actionBlock['stop_timestamp']
#       start_time = actionBlock['start_timestamp']
#       if stop_time == "None":
#           stop_time = float(start_time) + 1.0

#       start_time = datetime.utcfromtimestamp(start_time).strftime('%Y-%m-%d %H:%M:%S.%f')
#       stop_time = datetime.utcfromtimestamp(stop_time).strftime('%Y-%m-%d %H:%M:%S.%f')
#       spaker_name = actionBlock['speaker']['name']
#       actionText = participantsList[spaker_name] + ": " + actionBlock['action']

#       # Replace fixed participants named
#       for key,value in participantsList.items():
#         if key != "":
#           actionText = actionText.replace(key,value)
#       timelineList.append(dict(Task=actionText, Start=start_time, Finish=stop_time, Category = "Action"))


############ Skills ############
if show_skills == True:
  print("Showing skills")
  for skillBlock in structuredSkills['skillsTimeline']:
      start_time = skillBlock['start_timestamp']
      stop_time = skillBlock['stop_timestamp']
      if float(stop_time) < float(start_time) + 1.0:
          stop_time = float(start_time) + 3.0

      start_time = datetime.utcfromtimestamp(start_time).strftime('%Y-%m-%d %H:%M:%S.%f')
      stop_time = datetime.utcfromtimestamp(stop_time).strftime('%Y-%m-%d %H:%M:%S.%f')
      speaker_name = skillBlock['speaker']['name']
      actionText = participantsList[speaker_name] + ": " + skillBlock['main_skill'] + " - "+ skillBlock['skill_description']

      # Replace fixed participants named
      for key,value in participantsList.items():
        if key != "":
          actionText = actionText.replace(key,value)
      timelineList.append(dict(Task=actionText, Start=start_time, Finish=stop_time, Category = "Skill"))
      skillList.append(dict(Task=actionText, Start=start_time, Finish=stop_time, Category = "Skill"))
    


df = pd.DataFrame(sorted(timelineList, key=lambda x: x["Start"]))
# print(df)



Participants: 
dict_values(['John', 'Emily', 'Unknown', 'Captain Jackson', 'Unknown'])
dict_keys(['John', 'Emily', 'Unknown', 'Captain Jackson', 'None'])
Is John a participant? Y/N
Introduce the name of the participant to fix the translations: 
Is Emily a participant? Y/N
Is Captain Jackson a participant? Y/N
{'John': 'John', 'Emily': 'Emily', 'Unknown': 'Unknown', 'Captain Jackson': 'Captain Jackson', 'None': 'Unknown'}
Showing orders
Showing skills


C:\Users\Elsie\AppData\Local\Temp\ipykernel_18712\533752159.py:151: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  stop_time = datetime.utcfromtimestamp(stop_time).strftime('%Y-%m-%d %H:%M:%S.%f')
C:\Users\Elsie\AppData\Local\Temp\ipykernel_18712\533752159.py:152: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  start_time = datetime.utcfromtimestamp(orderBlock['start_timestamp']).strftime('%Y-%m-%d %H:%M:%S.%f')
C:\Users\Elsie\AppData\Local\Temp\ipykernel_18712\533752159.py:203: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware ob

In [17]:
with open(scenario+'_timeline_skills.json', 'w') as file:
    json.dump(skillList, file, indent=4)
with open(scenario+'_timeline_orders.json', 'w') as file:
    json.dump(ordersList, file, indent=4)
with open(scenario+'_timeline_total.json', 'w') as file:
    json.dump(timelineList, file, indent=4)

In [3]:


fig = px.timeline(df, x_start="Start", x_end="Finish", y="Task", category_orders=dict(Start=df["Start"].tolist()) )

colors = [
    'green' if category == "Presentation" else  # Presentation  - Green
    'red' if category == "Order" else           # Order  - Red
    '#ffd700' if category == "Action" else      # Action - Yellow
    'blue'                                      # Skill  - Blue
    for category in df["Category"]
]
fig.update_traces(marker_color=colors)

fig.update_yaxes(autorange="reversed") # otherwise tasks are listed from the bottom up
fig.update_layout(yaxis=dict(tickfont=dict(size=6)))
fig.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

--------------------------------------------------------------------------------------------------

------------------------------------------------------------------------------

Ingest into Neo4j DB

In [22]:
# https://sandbox.neo4j.com/

URI = 'bolt://34.227.21.28'
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "awards-clubs-pilots"
NEO4J_DATABASE = "neo4j"
AUTH = (NEO4J_USERNAME, NEO4J_PASSWORD)

In [23]:

from neo4j import GraphDatabase

neo4j_driver = GraphDatabase.driver(
    URI,
    auth = AUTH,
    notifications_min_severity = "OFF"
)

In [24]:
###### CREATE SPEAKERS
# for presentationBlock in structuredOrders['presentationsTimeline']:
#   speaker_id = presentationBlock["speaker"]["speaker_id"]
#   speaker_name = participantsList[presentationBlock["speaker"]["name"]]
#   role = presentationBlock["speaker"]["role"]

#   neo4j_driver.execute_query(
#       "MERGE (n:Speaker{speaker_id:'"+speaker_id+"',name:'"+speaker_name+"',role:'"+role+"'})"
#   )

# for particip in participantsList.values():
#   neo4j_driver.execute_query(
#       "MERGE (n:Speaker{name:'"+particip+"'})"
#   )


In [25]:
###### CREATE LINKS BETWEEN SPEAKERS BASED ON ORDERS
for orderBlock in structuredOrders['ordersTimeline']:
    start_time = orderBlock['start_timestamp'] #datetime.utcfromtimestamp(skillBlock['start_timestamp']).strftime('%Y-%m-%d %H:%M:%S.%f')
    stop_time = orderBlock['stop_timestamp'] #datetime.utcfromtimestamp(skillBlock['stop_timestamp']).strftime('%Y-%m-%d %H:%M:%S.%f')
    if stop_time == "None":
      stop_time = start_time + 1

    leader_name = participantsList[orderBlock['leader']['name']]

    # Create Speaker if doesn't exist
    if leader_name in participantsIds:
      speaker_id = participantsIds[leader_name]
    else:
      speaker_id = ""
    if leader_name in participantsRoles:
      role = participantsRoles[leader_name]
    else:
      role = ""
    neo4j_driver.execute_query(
        "MERGE (n:Speaker{speaker_id:'"+speaker_id+"',name:'"+leader_name+"',role:'"+role+"'})"
    )



    follower = orderBlock['follower']
    if follower != "None" and follower != "Unknown" and follower != None:
      follower_name = participantsList[orderBlock['follower']['name']]
    else:
      follower_name = 'Unknown'

    # Create Speaker if doesn't exist
    if follower_name in participantsIds:
      speaker_id = participantsIds[follower_name]
    else:
      speaker_id = ""
    if follower_name in participantsRoles:
      role = participantsRoles[follower_name]
    else:
      role = ""
    neo4j_driver.execute_query(
        "MERGE (n:Speaker{speaker_id:'"+speaker_id+"',name:'"+follower_name+"',role:'"+role+"'})"
    )


    order = orderBlock['action'].replace("'", "")
    # Replace fixed participants named
    for key,value in participantsList.items():
      if key != "":
        order = order.replace(key,value)

    # Relate Speaker to Action
    neo4j_driver.execute_query(
        "MATCH (sp1:Speaker {name: '"+leader_name+"'}), (sp2:Speaker {name: '"+follower_name+"'}) "+
        "MERGE (sp1)-[:ORDER_TO{from:"+str(start_time)+",to:"+str(stop_time)+",order:'"+order+"'}]->(sp2)"
    )








In [26]:
# ###### CREATE ACTIONS AND LINKS TO SPEAKERS
# for actionBlock in structuredActions['actionsTimeline']:
#     start_time = actionBlock['start_timestamp'] #datetime.utcfromtimestamp(skillBlock['start_timestamp']).strftime('%Y-%m-%d %H:%M:%S.%f')
#     stop_time = actionBlock['stop_timestamp'] #datetime.utcfromtimestamp(skillBlock['stop_timestamp']).strftime('%Y-%m-%d %H:%M:%S.%f')
#     if stop_time == "None":
#       stop_time = start_time + 1
#     speaker_name = participantsList[actionBlock['speaker']['name']]
#     action = actionBlock['action'].replace("'", "")

#     # Replace fixed participants named
#     for key,value in participantsList.items():
#       if key != "":
#         action = action.replace(key,value)
#     # Create Action
#     neo4j_driver.execute_query(
#         "MERGE (s:Action{name:'"+action+"'})"
#     )
#     # Relate Speaker to Action
#     neo4j_driver.execute_query(
#         "MATCH (ac:Action {name:'"+action+"'}), (sp:Speaker {name: '"+speaker_name+"'}) "+
#         "MERGE (sp)-[:HAS_DONE{from:"+str(start_time)+",to:"+str(stop_time)+"}]->(ac)"
#     )


In [27]:
###### CREATE SKILLS AND LINKS TO SPEAKERS
for skillBlock in structuredSkills['skillsTimeline']:
    start_time = skillBlock['start_timestamp'] #datetime.utcfromtimestamp(skillBlock['start_timestamp']).strftime('%Y-%m-%d %H:%M:%S.%f')
    stop_time = skillBlock['stop_timestamp'] #datetime.utcfromtimestamp(skillBlock['stop_timestamp']).strftime('%Y-%m-%d %H:%M:%S.%f')
    speaker_name = participantsList[skillBlock['speaker']['name']]
    main_skill = skillBlock['main_skill'].replace("'", "")

    skill_description = skillBlock['skill_description'].replace("'", "")
    # Replace fixed participants named
    for key,value in participantsList.items():
      if key != "":
        skill_description = skill_description.replace(key,value)

    sentence = skillBlock['sentence'].replace("'", "")
    # Replace fixed participants named
    for key,value in participantsList.items():
      if key != "":
        sentence = sentence.replace(key,value)

    # Create Main skill
    neo4j_driver.execute_query(
        "MERGE (s:Skill{name:'"+main_skill+"'})"
    )
    # Create Sub skill
    neo4j_driver.execute_query(
        "MERGE (ss:SubSkill{description:'"+skill_description+"'})"
    )
    # Relate Sub skill to main skill
    neo4j_driver.execute_query(
        "MATCH (ss:SubSkill {description:'"+skill_description+"'}), (s:Skill{name:'"+main_skill+"'}) "+
        "MERGE (ss)-[:RELATE_TO{from:"+str(start_time)+",to:"+str(stop_time)+"}]->(s)"
    )
    # # Relate Speaker to Main skill
    # neo4j_driver.execute_query(
    #     "MATCH (ss:Skill {name:'"+main_skill+"'}), (sp:Speaker {name: '"+speaker_name+"'}) "+
    #     "MERGE (sp)-[:HAS_SKILL{from:"+str(start_time)+",to:"+str(stop_time)+"}]->(ss)"
    # )
    # Relate Speaker to Sub skill
    neo4j_driver.execute_query(
        "MATCH (ss:SubSkill {description:'"+skill_description+"'}), (sp:Speaker {name: '"+speaker_name+"'}) "+
        "MERGE (sp)-[:HAS_SUBSKILL{from:"+str(start_time)+",to:"+str(stop_time)+",sentence:'"+sentence+"'}]->(ss)"
    )


In [ ]:
#   MATCH (s:Speaker)-[h]->(sk:Skill) RETURN s,sk
#   MATCH (s:Speaker)-[h]->(s2:Speaker) RETURN s,s2

In [ ]:

# a = neo4j_driver.execute_query(
#     "MATCH (n) RETURN DISTINCT n.type, n.name, n.id"
# )
# print(a)